# Tree Data Extraction and Analysis

This notebook demonstrates how to load, inspect, and extract tree positions and heights from the local GeoPackage (`.gpkg`) tree database (`Trees/125015_baeume.gpkg`).

In [1]:
import os
import pandas as pd
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
from shapely.geometry import box
from pyproj import Transformer

### 2. Path

In [2]:
TREE_GPKG_PATH = '../Trees/125015_baeume.gpkg'


### 3. Load the Tree Dataset

In [3]:
query_box = box(561000, 5505000, 575000, 5523000)
print("Loading tree dataset (this may take 10-15 seconds)...")
trees_gdf = gpd.read_file(TREE_GPKG_PATH, bbox=query_box)
print(f"Loaded {len(trees_gdf):,} trees.")
print("CRS of the dataset:", trees_gdf.crs)
trees_gdf.head()

Loading tree dataset (this may take 10-15 seconds)...
Loaded 928,531 trees.
CRS of the dataset: EPSG:25832


,dgmhoehe,baumhoehe,geometry
0,339.6,34.4,POINT (562018.5 5507559.1)
1,340.1,29.0,POINT (562009.9 5507560.7)
2,339.1,27.1,POINT (562025.3 5507562.7)
3,339.6,28.0,POINT (562011.7 5507567.5)
4,336.8,27.0,POINT (562049.5 5507568.5)


### Spatially Filter Trees for the Study Area

In [4]:
from shapely.geometry import box

# Define the GeoData study area bounding box
query_box = box(561000, 5505000, 575000, 5523000)

# Spatially filter trees within the study area box
trees_filtered = trees_gdf[trees_gdf.geometry.intersects(query_box)].copy()
print(f"Filtered {len(trees_filtered):,} trees inside the study area.")

Filtered 928,531 trees inside the study area.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
trees_filtered.plot(ax=ax, color='green', markersize=1, alpha=0.5, label='Trees')
gpd.GeoSeries([query_box], crs=trees_filtered.crs).boundary.plot(ax=ax, color='red', linewidth=2, label='Study Area')
ax.set_title("Trees inside GeoData study area")
plt.legend()
plt.show()

### 4. Extract Tree Features

In [ ]:
trees_filtered['Tree_Position_X'] = trees_filtered.geometry.x
trees_filtered['Tree_Position_Y'] = trees_filtered.geometry.y

columns_to_keep = ['Tree_Position_X', 'Tree_Position_Y', 'baumhoehe', 'dgmhoehe']
optional_keywords = ['crown', 'kronen', 'diameter', 'radius', 'shape', 'breite', 'durchmesser']
for col in trees_filtered.columns:
    if col.lower() != 'geometry' and col not in columns_to_keep:
        if any(keyword in col.lower() for keyword in optional_keywords):
            columns_to_keep.append(col)

trees_filtered = trees_filtered[columns_to_keep]
trees_filtered.head()

### 5. Inspect Tree Heights and Statistics

In [ ]:
print("Tree Height (baumhoehe) statistics (meters):")
print(trees_filtered['baumhoehe'].describe())

print("\nGround Elevation (dgmhoehe) statistics (meters):")
print(trees_filtered['dgmhoehe'].describe())

### 6. Output

In [ ]:
import os
import json
out_dir = r"E:\EnergyPraktikum_main\outputs"

trees_filtered.to_csv(os.path.join(out_dir, "wuerzburg_extracted_trees.csv"), index=False)

trees_filtered_dict = trees_filtered.to_dict(orient="records")
with open(os.path.join(out_dir, "wuerzburg_extracted_trees.json"), "w", encoding="utf-8") as f:
    json.dump(trees_filtered_dict, f)
